<a href="https://colab.research.google.com/github/Harshithpalan/Python-projects/blob/main/Multi-Task%20Learning%20Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Multi-Task Learning Model: All-in-One Classifier

This notebook will demonstrate the conceptual structure of a multi-task learning model designed to predict object type, weather, and scene from a single input. The core idea is to leverage a shared convolutional neural network (CNN) backbone to extract common features, followed by separate output layers (heads) for each specific task.

### Model Architecture Overview

1.  **Shared CNN Backbone**: This will be the feature extractor. For a real-world application, this would typically be a pre-trained model like ResNet, MobileNet, or EfficientNet. For this conceptual example, we'll use a simplified convolutional block.
2.  **Task-Specific Output Heads**: Each head will take the features from the backbone and produce predictions for its respective task (e.g., 'animal', 'vehicle', 'food' for object type; 'sunny', 'rainy' for weather; 'indoor', 'outdoor' for scene).

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.19.0


### 1. Define the Shared CNN Backbone

This part of the model will learn general visual features relevant to all tasks.

In [2]:
def build_shared_backbone(input_shape):
    inputs = keras.Input(shape=input_shape, name='input_image')

    # Simple convolutional blocks for feature extraction
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    return keras.Model(inputs=inputs, outputs=x, name='shared_cnn_backbone')

### 2. Define Task-Specific Output Heads

Each head will take the features from the backbone and make its specific prediction.

In [3]:
def build_object_type_head(backbone_output_features, num_object_types):
    x = layers.Dense(128, activation='relu')(backbone_output_features)
    x = layers.Dropout(0.3)(x)
    object_output = layers.Dense(num_object_types, activation='softmax', name='object_type_output')(x)
    return object_output

def build_weather_head(backbone_output_features, num_weather_types):
    x = layers.Dense(64, activation='relu')(backbone_output_features)
    x = layers.Dropout(0.2)(x)
    weather_output = layers.Dense(num_weather_types, activation='softmax', name='weather_output')(x)
    return weather_output

def build_scene_head(backbone_output_features, num_scene_types):
    x = layers.Dense(64, activation='relu')(backbone_output_features)
    x = layers.Dropout(0.2)(x)
    scene_output = layers.Dense(num_scene_types, activation='softmax', name='scene_output')(x)
    return scene_output

### 3. Assemble the Multi-Task Model

Now, we combine the shared backbone with all the task-specific heads.

In [4]:
# Define input shape (e.g., for 128x128 color images)
input_shape = (128, 128, 3)

# Define the number of classes for each task
num_object_types = 3  # e.g., animal, vehicle, food
num_weather_types = 2 # e.g., sunny, rainy
num_scene_types = 2   # e.g., indoor, outdoor

# Build the shared backbone
shared_backbone = build_shared_backbone(input_shape)

# Get the output features from the backbone
backbone_features = shared_backbone.output

# Build the heads
object_type_output = build_object_type_head(backbone_features, num_object_types)
weather_output = build_weather_head(backbone_features, num_weather_types)
scene_output = build_scene_head(backbone_features, num_scene_types)

# Create the final multi-task model
multi_task_model = keras.Model(
    inputs=shared_backbone.input,
    outputs=[object_type_output, weather_output, scene_output],
    name='all_in_one_classifier'
)

# Display the model summary
multi_task_model.summary()

Model: "all_in_one_classifier"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_image         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │        896 │ input_image[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 16, 16,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 32768)     │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │  8,388,864 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │     16,448 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │     16,448 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64)        │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ object_type_output  │ (None, 3)         │        387 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ weather_output      │ (None, 2)         │        130 │ dropout_2[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ scene_output        │ (None, 2)         │        130 │ dropout_3[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,548,551 (32.61 MB)

 Trainable params: 8,548,551 (32.61 MB)

 Non-trainable params: 0 (0.00 B)

### Training Considerations

To train this model, you would need a dataset where each image has **all three labels**: object type, weather, and scene.

When compiling the model, you would specify a loss function for each output and potentially assign weights to each loss if some tasks are more critical than others or if their error magnitudes differ significantly.

```python
multi_task_model.compile(
    optimizer='adam',
    loss={
        'object_type_output': 'categorical_crossentropy',
        'weather_output': 'categorical_crossentropy',
        'scene_output': 'categorical_crossentropy'
    },
    loss_weights={
        'object_type_output': 1.0, # You can adjust these weights
        'weather_output': 0.8,
        'scene_output': 0.8
    },
    metrics={
        'object_type_output': ['accuracy'],
        'weather_output': ['accuracy'],
        'scene_output': ['accuracy']
    }
)

# Example of how to fit the model (requires actual data)
# history = multi_task_model.fit(
#     x=your_image_data,
#     y={'object_type_output': your_object_labels,
#        'weather_output': your_weather_labels,
#        'scene_output': your_scene_labels},
#     epochs=10,
#     batch_size=32,
#     validation_data=(validation_image_data,
#                      {'object_type_output': validation_object_labels,
#                       'weather_output': validation_weather_labels,
#                       'scene_output': validation_scene_labels})
# )
```

This setup allows the model to learn shared representations that are useful for all three tasks simultaneously, potentially leading to better generalization and efficiency compared to training three separate models.